---
## ROUGE Evaluation
Comprehensive ROUGE-based evaluation of the fine-tuned T5 model against the held-out test set.
Covers **ROUGE-1**, **ROUGE-2**, **ROUGE-L**, and **ROUGE-Lsum**, with per-sample scoring,
aggregate statistics, per-action-type breakdown, and score-distribution visualisations.

All outputs are saved to a dedicated `rouge_evaluation/` subfolder inside `model_output_dir`.

### Step 1 — Install ROUGE Dependencies

In [ ]:
%pip install -q rouge-score evaluate nltk

import os

# Dedicated output folder for all ROUGE artefacts
base_dir  = model_output_dir if "model_output_dir" in globals() else "."
rouge_dir = os.path.join(base_dir, "rouge_evaluation")
os.makedirs(rouge_dir, exist_ok=True)

print("[Step 1] Dependencies installed.")
print(f"[Step 1] All ROUGE outputs will be saved to: {rouge_dir}")

### Step 2 — Compute Per-Sample ROUGE Scores (All Variants)

In [ ]:
from rouge_score import rouge_scorer
import pandas as pd

# ------------------------------------------------------------------
# Requires eval_results_df in scope (produced by 'Test Model' cell).
# Columns needed: 'input', 'target', 'predicted'
# ------------------------------------------------------------------

ROUGE_METRICS = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

scorer = rouge_scorer.RougeScorer(ROUGE_METRICS, use_stemmer=True)

records = []
for _, row in eval_results_df.iterrows():
    reference  = row["target"].strip()
    prediction = row["predicted"].strip()
    scores = scorer.score(reference, prediction)
    record = {
        "input":     row["input"],
        "target":    reference,
        "predicted": prediction,
    }
    for metric in ROUGE_METRICS:
        record[f"{metric}_precision"] = scores[metric].precision
        record[f"{metric}_recall"]    = scores[metric].recall
        record[f"{metric}_f1"]        = scores[metric].fmeasure
    records.append(record)

rouge_df = pd.DataFrame(records)

print(f"[Step 2] Scored {len(rouge_df)} test examples across {len(ROUGE_METRICS)} ROUGE variants.")
print(f"[Step 2] Columns produced: {[f'{m}_(precision/recall/f1)' for m in ROUGE_METRICS]}")
print("\n[Step 2] Sample output (first 5 rows, F1 columns):")
f1_cols = ["target", "predicted"] + [f"{m}_f1" for m in ROUGE_METRICS]
print(rouge_df[f1_cols].head(5).to_string(index=False))

### Step 3 — Aggregate ROUGE Scores (Mean, Median, Std, Min, Max)

In [ ]:
import numpy as np

summary_rows = []
for metric in ROUGE_METRICS:
    for stat_name, fn in [("mean", np.mean), ("median", np.median),
                          ("std",  np.std),  ("min", np.min), ("max", np.max)]:
        for component in ["precision", "recall", "f1"]:
            col = f"{metric}_{component}"
            summary_rows.append({
                "metric":    metric,
                "component": component,
                "statistic": stat_name,
                "value":     fn(rouge_df[col]),
            })

summary_df = pd.DataFrame(summary_rows)
pivot = summary_df.pivot_table(
    index=["metric", "component"],
    columns="statistic",
    values="value"
)[["mean", "median", "std", "min", "max"]]

print("[Step 3] ROUGE AGGREGATE SCORES — Precision / Recall / F1")
print("=" * 70)
print(pivot.round(4).to_string())

print("\n[Step 3] F1 SUMMARY (mean ± std  |  min → max)")
print("-" * 55)
for metric in ROUGE_METRICS:
    col    = f"{metric}_f1"
    mean_f1   = rouge_df[col].mean()
    std_f1    = rouge_df[col].std()
    min_f1    = rouge_df[col].min()
    max_f1    = rouge_df[col].max()
    print(f"  {metric:<12}  {mean_f1:.4f} ± {std_f1:.4f}  |  {min_f1:.4f} → {max_f1:.4f}")

print(f"\n[Step 3] Computed over {len(rouge_df)} test samples.")

### Step 4 — ROUGE Scores Breakdown by Action Type

In [ ]:
rouge_df["action_type"] = rouge_df["input"].str.extract(r"action: (\w+)")

f1_cols_only    = [f"{m}_f1" for m in ROUGE_METRICS]
by_action_rouge = (
    rouge_df.groupby("action_type")[f1_cols_only]
    .mean()
    .round(4)
    .reset_index()
)
count_per_action = rouge_df["action_type"].value_counts().rename("sample_count")
by_action_rouge  = by_action_rouge.merge(count_per_action, on="action_type")

print("[Step 4] MEAN F1 ROUGE SCORES BY ACTION TYPE")
print("=" * 75)
print(by_action_rouge.to_string(index=False))
print(f"\n[Step 4] {len(by_action_rouge)} distinct action types evaluated.")

weak_actions = by_action_rouge[
    by_action_rouge["rougeL_f1"] < 0.5
]["action_type"].tolist()
if weak_actions:
    print(f"[Step 4] WARNING — Action types with ROUGE-L F1 < 0.50: {weak_actions}")
else:
    print("[Step 4] All action types scored ROUGE-L F1 >= 0.50.")

### Step 5 — ROUGE F1 Score Distributions (Histograms)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes   = axes.flatten()
colors = ["steelblue", "darkorange", "seagreen", "mediumpurple"]

print("[Step 5] ROUGE F1 Distribution Summary")
print("=" * 60)

for i, metric in enumerate(ROUGE_METRICS):
    col    = f"{metric}_f1"
    values = rouge_df[col]
    ax     = axes[i]

    ax.hist(values, bins=20, color=colors[i], edgecolor="white", alpha=0.85)
    ax.axvline(values.mean(),   color="red",   linestyle="--", linewidth=1.5,
               label=f"Mean   {values.mean():.3f}")
    ax.axvline(values.median(), color="black", linestyle=":",  linewidth=1.5,
               label=f"Median {values.median():.3f}")
    ax.set_title(f"{metric.upper()} F1 Distribution", fontsize=12, fontweight="bold")
    ax.set_xlabel("F1 Score")
    ax.set_ylabel("Count")
    ax.set_xlim(0, 1)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    pct_above_08 = (values >= 0.8).mean() * 100
    pct_below_05 = (values <  0.5).mean() * 100
    print(f"  {metric.upper():<12}  mean={values.mean():.4f}  median={values.median():.4f}  "
          f">=0.80: {pct_above_08:.1f}%  <0.50: {pct_below_05:.1f}%")

plt.suptitle("ROUGE F1 Score Distributions on Test Set",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()

dist_path = os.path.join(rouge_dir, "rouge_distributions.png")
plt.savefig(dist_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n[Step 5] Histogram saved to: {dist_path}")

### Step 6 — ROUGE Scores by Action Type (Bar Chart)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

action_types = by_action_rouge["action_type"].tolist()
x          = np.arange(len(action_types))
bar_width   = 0.2
colors      = ["steelblue", "darkorange", "seagreen", "mediumpurple"]

fig, ax = plt.subplots(figsize=(max(10, len(action_types) * 1.8), 5))

print("[Step 6] ROUGE Mean F1 per Action Type")
print("=" * 60)

for i, metric in enumerate(ROUGE_METRICS):
    col    = f"{metric}_f1"
    offset = (i - len(ROUGE_METRICS) / 2 + 0.5) * bar_width
    ax.bar(x + offset, by_action_rouge[col], width=bar_width,
           label=metric.upper(), color=colors[i], alpha=0.85, edgecolor="white")

    scores_str = "  ".join(
        f"{act}={val:.3f}"
        for act, val in zip(action_types, by_action_rouge[col])
    )
    print(f"  {metric.upper():<12}  {scores_str}")

ax.set_xticks(x)
ax.set_xticklabels(action_types, rotation=30, ha="right")
ax.set_xlabel("Action Type")
ax.set_ylabel("Mean F1 Score")
ax.set_title("Mean ROUGE F1 Scores by Action Type", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
bar_path = os.path.join(rouge_dir, "rouge_by_action_type.png")
plt.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n[Step 6] Bar chart saved to: {bar_path}")

### Step 7 — Low-Scoring Examples (ROUGE-L F1 < 0.5)

In [ ]:
THRESHOLD = 0.5

low_rouge = (
    rouge_df[rouge_df["rougeL_f1"] < THRESHOLD]
    .sort_values("rougeL_f1")
    [["input", "target", "predicted",
      "rouge1_f1", "rouge2_f1", "rougeL_f1", "rougeLsum_f1"]]
    .reset_index(drop=True)
)

total   = len(rouge_df)
n_low   = len(low_rouge)
pct_low = n_low / total * 100

print(f"[Step 7] Low-Scoring Examples — ROUGE-L F1 < {THRESHOLD}")
print("=" * 80)
print(f"[Step 7] {n_low} / {total} examples below threshold ({pct_low:.1f}%)")
print(f"[Step 7] {total - n_low} / {total} examples at or above threshold "
      f"({100 - pct_low:.1f}%)")
print()

if low_rouge.empty:
    print("[Step 7] No low-scoring examples — all predictions scored ROUGE-L F1 >= 0.50.")
else:
    for idx, row in low_rouge.iterrows():
        print(f"  [{idx + 1}/{n_low}] Input:     {row['input']}")
        print(f"          Target:    {row['target']}")
        print(f"          Predicted: {row['predicted']}")
        print(f"          ROUGE-1={row['rouge1_f1']:.4f}  "
              f"ROUGE-2={row['rouge2_f1']:.4f}  "
              f"ROUGE-L={row['rougeL_f1']:.4f}  "
              f"ROUGE-Lsum={row['rougeLsum_f1']:.4f}")
        print("-" * 80)

### Step 8 — Cross-Validation with HuggingFace `evaluate` Library

In [ ]:
import evaluate

hf_rouge = evaluate.load("rouge")

predictions = rouge_df["predicted"].tolist()
references  = rouge_df["target"].tolist()

hf_results = hf_rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True,
    use_aggregator=True,
)

print("[Step 8] HuggingFace evaluate — Corpus-Level ROUGE (mean F1)")
print("=" * 55)
for key, value in sorted(hf_results.items()):
    print(f"  {key:<15} {value:.4f}")

print("\n[Step 8] Cross-library sanity check (rouge_score vs hf evaluate):")
print("-" * 60)
for metric in ROUGE_METRICS:
    rs_val = rouge_df[f"{metric}_f1"].mean()
    hf_val = hf_results.get(metric, float("nan"))
    delta  = abs(rs_val - hf_val)
    status = "OK" if delta < 0.005 else "CHECK"
    print(f"  [{status}] {metric:<12}  rouge_score={rs_val:.4f}  "
          f"hf_evaluate={hf_val:.4f}  delta={delta:.4f}")

print(f"\n[Step 8] Evaluated on {len(predictions)} predictions.")

### Step 9 — Save All ROUGE Results to `rouge_evaluation/` Folder

In [ ]:
import json

# --- 1. Per-sample scores CSV ---
csv_path = os.path.join(rouge_dir, "rouge_per_sample.csv")
rouge_df.to_csv(csv_path, index=False)
print(f"[Step 9] Per-sample ROUGE scores   → {csv_path}")
print(f"         Rows: {len(rouge_df)}  |  Columns: {list(rouge_df.columns)}")

# --- 2. Per-action-type breakdown CSV ---
action_csv_path = os.path.join(rouge_dir, "rouge_by_action_type.csv")
by_action_rouge.to_csv(action_csv_path, index=False)
print(f"\n[Step 9] Per-action-type breakdown → {action_csv_path}")
print(f"         Action types: {by_action_rouge['action_type'].tolist()}")

# --- 3. Aggregate summary JSON ---
aggregate_summary = {"n_samples": len(rouge_df), "metrics": {}}
for metric in ROUGE_METRICS:
    col = f"{metric}_f1"
    aggregate_summary["metrics"][metric] = {
        "mean_f1":   round(rouge_df[col].mean(),   4),
        "median_f1": round(rouge_df[col].median(), 4),
        "std_f1":    round(rouge_df[col].std(),    4),
        "min_f1":    round(rouge_df[col].min(),    4),
        "max_f1":    round(rouge_df[col].max(),    4),
    }
aggregate_summary["hf_evaluate_corpus_f1"] = {
    k: round(v, 4) for k, v in hf_results.items()
}
aggregate_summary["low_scoring_examples"] = {
    "threshold_rougeL_f1": THRESHOLD,
    "count":      int(n_low),
    "percentage": round(pct_low, 2),
}

json_path = os.path.join(rouge_dir, "rouge_summary.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(aggregate_summary, f, indent=2)
print(f"\n[Step 9] Aggregate summary JSON    → {json_path}")

# --- Final checklist ---
print("\n" + "=" * 65)
print("[Step 9] ROUGE EVALUATION COMPLETE — FILE CHECKLIST")
print("=" * 65)
saved_files = [
    ("rouge_per_sample.csv",       "Per-sample P / R / F1 for all ROUGE variants"),
    ("rouge_by_action_type.csv",   "Mean F1 per action type"),
    ("rouge_summary.json",         "Aggregate stats + HF cross-check + low-score summary"),
    ("rouge_distributions.png",    "F1 histogram (2x2 grid, all variants)"),
    ("rouge_by_action_type.png",   "Grouped bar chart by action type"),
]
for fname, desc in saved_files:
    fpath  = os.path.join(rouge_dir, fname)
    status = "OK" if os.path.exists(fpath) else "MISSING"
    print(f"  [{status}] {fname:<35} {desc}")

print(f"\n[Step 9] Output folder: {rouge_dir}")
print("\n[Step 9] Aggregate summary:")
print(json.dumps(aggregate_summary, indent=2))